# ML-based mapping of post-harvest residual soil nitrate

This notebook reproduces the Random Forest, XGBoost, and stacked-ensemble workflow used for the patchCROP residual nitrate analysis.

**Repository inputs**
- `data/patchCROP_Residual_Nitrate.xlsx` — analysis dataset (with README/data dictionary sheet)
- `data/spatial/30patches.zip` — patch boundary shapefile archive

Run the notebook from the repository root or from the `notebooks/` directory. Generated tables and figures are written to `outputs/` and are intentionally excluded from version control.


In [ ]:

pip install --upgrade pip

In [ ]:
pip install PyALE

In [ ]:
# ======================================
# BLOCK A. GLOBAL SETUP & REPRODUCIBILITY (UPDATED)
# ======================================

import os, re, zipfile, warnings, json, math, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, precision_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer

from xgboost import XGBRegressor

import shap
import geopandas as gpd

%matplotlib widget

# ---- Reproducibility
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ---- Reduce noise
warnings.filterwarnings("once")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("\n" + "="*100)
print("BLOCK A — GLOBAL SETUP COMPLETE (WINDOWS LOCAL)")
print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("RANDOM_STATE     :", RANDOM_STATE)
print("="*100)


In [ ]:
# ======================================
# BLOCK B. PATHS + LOAD DATA + CLEAN TYPES (UPDATED PATHS)
# ======================================

from pathlib import Path

# Resolve repository root whether the notebook is launched from the repo root
# or directly from the notebooks/ directory.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

BASE_DIR = REPO_ROOT
DATA_PATH = REPO_ROOT / "data" / "patchCROP_Residual_Nitrate.xlsx"
BOUNDARY_ZIP = REPO_ROOT / "data" / "spatial" / "30patches.zip"

OUT_DIR   = REPO_ROOT / "outputs"
PLOTS_DIR = OUT_DIR / "plots"
EXCEL_OUT = OUT_DIR / "RiskMapping_Results.xlsx"

for d in [OUT_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\n" + "="*80)
print("BLOCK B — PATHS INITIALIZED")
print("REPO_ROOT:", REPO_ROOT)
print("DATA     :", DATA_PATH)
print("BOUNDARY :", BOUNDARY_ZIP)
print("OUT_DIR  :", OUT_DIR)
print("PLOTS    :", PLOTS_DIR)
print("EXCEL    :", EXCEL_OUT)
print("="*80)

# ---- Load dataset
df = pd.read_excel(DATA_PATH)
print("\nInitial dataset shape:", df.shape)

# ---- Type coercion
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Patch"] = df["Patch"].astype(str)
df["Subsample"] = df["Subsample"].astype(str)

cat_cols = [
    "Crop_during_sampling", "Crop_group_during_sampling",
    "Crop_2_3_month_before_sampling", "Crop_group_2_3_month_b4_sampling",
    "Crop_season", "Campaign",
]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype("category")

# ---- Filter AH campaigns only
df = df[df["Campaign"].astype(str).str.startswith("AH")].copy()
print("\nAfter filtering AH campaigns:", df.shape)
print("Unique campaigns:", sorted(df["Campaign"].astype(str).unique()))

# ---- Drop rows with missing coordinates
df = df.dropna(subset=["X", "Y"])
print("After dropping missing X/Y:", df.shape)


In [ ]:
# ======================================
# BLOCK C. SPATIAL PATCH HOLD-OUT SPLIT (UNCHANGED)
# ======================================

TARGET_DEPTH = "0_30"
TARGET_COL = f"kgNO3_{TARGET_DEPTH}"
print("\nTarget variable:", TARGET_COL)

df_d = df[~df[TARGET_COL].isna()].copy()
print("Rows with observed target:", df_d.shape)

patch_xy = (
    df_d.groupby("Patch")[["X", "Y"]]
    .mean()
    .dropna()
)

all_patches = patch_xy.index.astype(str).to_numpy()
print("Total patches with data:", len(all_patches))

TEST_PATCHES = np.array(['13','39','51', '65', '68', '96', '105','119'], dtype=str)

missing = np.setdiff1d(TEST_PATCHES, all_patches)
if len(missing) > 0:
    raise ValueError(f"These test patches are not found in the dataset: {missing}")

TRAIN_PATCHES = np.setdiff1d(all_patches, TEST_PATCHES)

print("\nTrain patches:", len(TRAIN_PATCHES))
print("Test patches :", len(TEST_PATCHES))
print("Test patch IDs:", TEST_PATCHES)

pd.DataFrame({"Patch": TRAIN_PATCHES}).to_csv(os.path.join(OUT_DIR, "train_patches.csv"), index=False)
pd.DataFrame({"Patch": TEST_PATCHES}).to_csv(os.path.join(OUT_DIR, "test_patches.csv"), index=False)

df_train = df_d[df_d["Patch"].isin(TRAIN_PATCHES)].copy()
df_test  = df_d[df_d["Patch"].isin(TEST_PATCHES)].copy()

print("\nTrain rows:", df_train.shape)
print("Test rows :", df_test.shape)

print("\nTrain patch sample:", TRAIN_PATCHES[:22])
print("Test patch sample :", TEST_PATCHES[:10])


In [ ]:
# ======================================
# BLOCK C-2. TRAIN vs TEST PATCH MAP (ROBUST)
# ======================================

print("\n" + "="*80)
print("BLOCK C-2 — VISUALIZING TRAIN / TEST PATCH SPLIT")
print("="*80)

import geopandas as gpd
import matplotlib.pyplot as plt

# ---- Extract boundary zip
tmp_boundary_dir = os.path.join(OUT_DIR, "boundary_preview")
os.makedirs(tmp_boundary_dir, exist_ok=True)

with zipfile.ZipFile(BOUNDARY_ZIP, "r") as zf:
    zf.extractall(tmp_boundary_dir)

# ---- Recursively find shapefile
shp_path = None
for root, dirs, files in os.walk(tmp_boundary_dir):
    for f in files:
        if f.endswith(".shp"):
            shp_path = os.path.join(root, f)
            break
    if shp_path:
        break

if shp_path is None:
    raise FileNotFoundError("No .shp file found anywhere inside boundary zip.")

print("Using shapefile:", shp_path)

# ---- Load boundaries
gdf_patch = gpd.read_file(shp_path)
gdf_patch["Patch"] = gdf_patch["Patch"].astype(str)

# ---- Assign split label
gdf_patch["Split"] = np.where(
    gdf_patch["Patch"].isin(TEST_PATCHES),
    "Test",
    np.where(gdf_patch["Patch"].isin(TRAIN_PATCHES), "Train", "Unused")
)

print("\nPatch split counts:")
print(gdf_patch["Split"].value_counts())

# ---- Plot
fig, ax = plt.subplots(figsize=(10, 8))

gdf_patch[gdf_patch["Split"] == "Train"].plot(
    ax=ax, color="lightgray", edgecolor="black", label="Train"
)

gdf_patch[gdf_patch["Split"] == "Test"].plot(
    ax=ax, color="red", edgecolor="black", label="Test"
)

# ---- Label patch IDs
for _, row in gdf_patch.iterrows():
    if row["Split"] != "Unused":
        c = row.geometry.centroid
        ax.text(
            c.x, c.y, row["Patch"],
            fontsize=8,
            ha="center", va="center",
            color="white" if row["Split"] == "Test" else "black",
            weight="bold"
        )

ax.set_title(
    "Spatial Patch Hold-Out Validation\n(Test patches highlighted in red)",
    fontsize=14
)
ax.axis("off")
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "patch_train_test_map.png"), dpi=300)
plt.close()

print("✅ Train/Test patch map saved to:", PLOTS_DIR)


In [ ]:
# ======================================
# BLOCK D. FEATURE DEFINITIONS (MINOR: define CAT_VARS)
# ======================================

STATIC_COMMON = [
    "Sand", "Silt", "Clay",
    "DEM", "Slope", "Aspect", "TWI", "X", "Y",
    "N_surplus", "Fert_N_curr_year",
    "Crop_during_sampling",
]

STATIC_BY_DEPTH = {
    "0_30":  ["Ct_0_30", "Ct_30_60", "Ct_60_90"],
    "30_60": ["BD_0_30cm", "BD_30_60cm", "Ct_30_60", "Ct_60_90"],
    "60_90": ["kgNO3_0_30", "Ct_0_30", "Ct_30_60", "Ct_60_90"],
}

STATIC_VARS = STATIC_COMMON + STATIC_BY_DEPTH.get(TARGET_DEPTH, [])
STATIC_VARS = [c for c in STATIC_VARS if c in df_train.columns]

print("\nStatic variables found:", len(STATIC_VARS))
print("Static vars sample:", STATIC_VARS[:15])

WINDOWS = ["1d","3d", "7d", "15d", "30d", "60d", "90d"]

DYNAMIC_PATTERNS = [
    r"^Precipitation_sum_",
    r"^R_profile_mm_sum_",
    r"^Tmin_mean_",
    r"^Tmax_mean_",
    r"^NDVI_mean_",
    r"^NDRE_mean_",
    r"^SAVI_mean_",
    r"^SWIR 1_mean_",
    r"^SWIR 2_mean_",
    r"^NIR_mean_",
    r"^NDWI_mean_",
    r"^dS_30cm_mm_mean_",
    r"^WC_30cm_mean_",
    r"^dS_total_mm_mean_",
    r"^WC_60cm_mean_",
    r"^WC_90cm_mean_",
    r"^dS_60cm_mm_mean_",
    r"^dS_90cm_mm_mean_",
] #r"^R_profile_pos_mm_sum_",

def get_window_vars(columns, window, patterns):
    base = "(" + "|".join(patterns) + ")"
    regex = re.compile(base + window + r"$")
    return [c for c in columns if regex.search(c)]

print("\nAvailable rolling windows:", WINDOWS)
print("Example dynamic vars for 30d:", get_window_vars(df_train.columns, "30d", DYNAMIC_PATTERNS)[:10])

# Categorical variables you want to use (expand if needed)
CAT_VARS_MASTER = [
    "Crop_group_during_sampling",
    "Crop_group_2_3_month_b4_sampling",
    "Crop_season",
    "Crop_during_sampling",
    "Crop_2_3_month_before_sampling",
]
CAT_VARS_MASTER = [c for c in CAT_VARS_MASTER if c in df_train.columns]
print("\nCategorical variables available:", CAT_VARS_MASTER)


In [ ]:
# ======================================
# BLOCK E. METRICS & RISK DEFINITIONS (UNCHANGED)
# ======================================

def regression_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

def risk_class_from_pred(y_pred):
    q25, q50, q75 = np.percentile(y_pred, [25, 50, 75])
    bins = [-np.inf, q25, q50, q75, np.inf]
    labels = ["Low", "Moderate", "High", "Extreme"]
    cls = pd.cut(y_pred, bins=bins, labels=labels, include_lowest=True)
    return cls, {"q25":q25, "q50":q50, "q75":q75}

def auc_precision_extreme(y_true, y_score, pred_q75):
    q75_true = np.percentile(y_true, 75)
    y_true_ext = (y_true >= q75_true).astype(int)
    y_pred_ext = (y_score >= pred_q75).astype(int)

    auc  = roc_auc_score(y_true_ext, y_score)
    prec = precision_score(y_true_ext, y_pred_ext, zero_division=0)
    return float(auc), float(prec), float(q75_true)

print("\n" + "="*80)
print("BLOCK E — METRICS & RISK FUNCTIONS READY")
print("="*80)


In [ ]:
# ======================================
# BLOCK F. MISSING-AWARE + CATEGORICAL-AWARE PIPELINE (UPDATED)
# ======================================

def add_missing_flags(X):
    flags = X.isna().astype(int)
    flags.columns = [f"{c}_isNA" for c in X.columns]
    return pd.concat([X, flags], axis=1)

def build_pipeline(model, numeric_cols, categorical_cols):
    """
    Pipeline:
      - Numeric: missing flags + median impute
      - Categorical: most_frequent impute + one-hot encode (handle_unknown=ignore)
      - Model
    """
    num_pipe = Pipeline([
        ("missing_flags", FunctionTransformer(add_missing_flags, validate=False)),
        ("imputer", SimpleImputer(strategy="median")),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])

def get_preprocessed_feature_names(fitted_pipe):
    """
    Returns feature names after preprocessing (numeric + isNA + one-hot).
    Works after the pipeline is fitted.
    """
    prep = fitted_pipe.named_steps["prep"]

    names = []
    for name, trans, cols in prep.transformers_:
        if name == "num":
            base = list(cols)
            names.extend(base)
            names.extend([f"{c}_isNA" for c in base])
        elif name == "cat":
            ohe = trans.named_steps["onehot"]
            names.extend(list(ohe.get_feature_names_out(cols)))
    return np.array(names, dtype=object)

print("\n" + "="*80)
print("BLOCK F — MIXED-TYPE PIPELINE BUILDER READY (NUM + CAT)")
print("="*80)


In [ ]:
# ======================================
# BLOCK G. OPTUNA OBJECTIVE FACTORY (UPDATED for NUM+CAT pipeline)
# ======================================

def objective_rf(trial, X, y, groups, window_name, cv_log, numeric_cols, categorical_cols):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=100),
        "max_depth": trial.suggest_categorical("max_depth", [None, 2, 8]),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.3, 1.0]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 8),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }
    params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0) if params["bootstrap"] else None

    pipe = build_pipeline(
        RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=1),
        numeric_cols, categorical_cols
    )

    gkf = GroupKFold(n_splits=5)
    rmses = []

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[va])
        rmse = np.sqrt(mean_squared_error(y.iloc[va], pred))
        rmses.append(rmse)

        cv_log.append({
            "window": window_name, "model": "rf", "trial": trial.number,
            "fold": fold, "rmse": float(rmse), **params
        })

    return float(np.mean(rmses))

def objective_xgb(trial, X, y, groups, window_name, cv_log, numeric_cols, categorical_cols):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 5.0, 10.0, log=True),
    }

    pipe = build_pipeline(
        XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            predictor="cpu_predictor",
        ),
        numeric_cols, categorical_cols
    )

    gkf = GroupKFold(n_splits=5)
    rmses = []

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[va])
        rmse = np.sqrt(mean_squared_error(y.iloc[va], pred))
        rmses.append(rmse)

        cv_log.append({
            "window": window_name, "model": "xgb", "trial": trial.number,
            "fold": fold, "rmse": float(rmse), **params
        })

    return float(np.mean(rmses))

print("\n" + "="*80)
print("BLOCK G — OPTUNA OBJECTIVES READY (NUM + CAT)")
print("="*80)


In [ ]:
# ======================================
# BLOCK H. MAIN ROLLING-WINDOW TRAINING (UPDATED: use CAT + OOF stacking)
# ======================================

results_rows = []
cv_log = []
meta_weights = []
test_pred_rows = []

best_bundle_by_window = {}

N_TRIALS = 300

for w in WINDOWS:
    print("\n" + "="*100)
    print(f"WINDOW: {w}")
    print("="*100)

    # ---- 1) Feature selection
    dyn_vars = get_window_vars(df_train.columns, w, DYNAMIC_PATTERNS)
    feature_cols = [c for c in (STATIC_VARS + dyn_vars) if c in df_train.columns]

    if len(feature_cols) == 0:
        print("⛔ No features for this window. Skipping.")
        continue

    # Keep ALL columns (numeric + categorical) here
    X_train = df_train[feature_cols].copy()
    X_test  = df_test[feature_cols].copy()
    y_train = df_train[TARGET_COL]
    y_test  = df_test[TARGET_COL]
    groups  = df_train["Patch"].values

    # Define numeric vs categorical columns (per-window)
    numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
    categorical_cols = [c for c in CAT_VARS_MASTER if (c in feature_cols and c in X_train.columns)]

    print(f"Train rows={len(X_train)} | Test rows={len(X_test)} | "
          f"NUM={len(numeric_cols)} | CAT={len(categorical_cols)} | "
          f"Total raw cols={len(feature_cols)}")

    if len(numeric_cols) == 0 and len(categorical_cols) == 0:
        print("⛔ No usable numeric/categorical columns for this window. Skipping.")
        continue

    # ---- 2) Optuna RF
    print("\n-> Tuning RF...")
    study_rf = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_STATE))
    study_rf.optimize(
        lambda t: objective_rf(t, X_train, y_train, groups, w, cv_log, numeric_cols, categorical_cols),
        n_trials=N_TRIALS
    )

    best_rf = build_pipeline(
        RandomForestRegressor(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=1),
        numeric_cols, categorical_cols
    )
    best_rf.fit(X_train, y_train)

    print("\nBest RF parameters:")
    for k, v in study_rf.best_params.items():
        print(f"   {k}: {v}")
    print("   RF CV RMSE:", round(study_rf.best_value, 4))

    # ---- 3) Optuna XGB
    print("\n-> Tuning XGB...")
    study_xgb = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_STATE))
    study_xgb.optimize(
        lambda t: objective_xgb(t, X_train, y_train, groups, w, cv_log, numeric_cols, categorical_cols),
        n_trials=N_TRIALS
    )

    best_xgb = build_pipeline(
        XGBRegressor(
            **study_xgb.best_params,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            predictor="cpu_predictor",
        ),
        numeric_cols, categorical_cols
    )
    best_xgb.fit(X_train, y_train)

    print("\nBest XGB parameters:")
    for k, v in study_xgb.best_params.items():
        print(f"   {k}: {v}")
    print("   XGB CV RMSE:", round(study_xgb.best_value, 4))

    # ---- 4) OOF Stacking ensemble (NO LEAKAGE)
    print("\n-> OOF Stacking (GroupKFold) ...")
    gkf_stack = GroupKFold(n_splits=5)

    oof_rf  = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))

    for fold, (tr, va) in enumerate(gkf_stack.split(X_train, y_train, groups), 1):
        rf_fold  = clone(best_rf)
        xgb_fold = clone(best_xgb)

        rf_fold.fit(X_train.iloc[tr], y_train.iloc[tr])
        xgb_fold.fit(X_train.iloc[tr], y_train.iloc[tr])

        oof_rf[va]  = rf_fold.predict(X_train.iloc[va])
        oof_xgb[va] = xgb_fold.predict(X_train.iloc[va])

    meta_model = Ridge(alpha=1.0)  # Ridge in sklearn uses deterministic solvers here
    meta_model.fit(np.column_stack([oof_rf, oof_xgb]), y_train)

    print("\nMeta-model coefficients (trained on OOF preds):")
    for name, coef in zip(["rf", "xgb"], meta_model.coef_):
        print(f"   {name}: {coef:.4f}")
        meta_weights.append({"window": w, "model": name, "coef": float(coef)})

    # Refit base models on full training
    best_rf.fit(X_train, y_train)
    best_xgb.fit(X_train, y_train)

    # ---- 5) Test predictions
    yhat_rf  = best_rf.predict(X_test)
    yhat_xgb = best_xgb.predict(X_test)
    yhat_ens = meta_model.predict(np.column_stack([yhat_rf, yhat_xgb]))
    unc_ens  = np.std(np.column_stack([yhat_rf, yhat_xgb]), axis=1)

    rf_rmse, rf_mae, rf_r2 = regression_metrics(y_test, yhat_rf)
    xg_rmse, xg_mae, xg_r2 = regression_metrics(y_test, yhat_xgb)
    en_rmse, en_mae, en_r2 = regression_metrics(y_test, yhat_ens)

    risk_cls, q_pred = risk_class_from_pred(yhat_ens)
    auc, prec, _ = auc_precision_extreme(y_test.values, yhat_ens, q_pred["q75"])

    print(f"\nENS TEST PERFORMANCE ({w})")
    print(f"   RF  : RMSE={rf_rmse:.3f}  R2={rf_r2:.3f}")
    print(f"   XGB : RMSE={xg_rmse:.3f}  R2={xg_r2:.3f}")
    print(f"   ENS : RMSE={en_rmse:.3f}  R2={en_r2:.3f}")
    print(f"   AUC (extreme)={auc:.3f}  Precision={prec:.3f}")

    results_rows.append({
        "window": w,
        "rf_rmse": rf_rmse, "rf_r2": rf_r2,
        "xgb_rmse": xg_rmse, "xgb_r2": xg_r2,
        "ens_rmse": en_rmse, "ens_r2": en_r2,
        "auc_extreme": auc, "precision_extreme": prec,
        "rf_best_params": json.dumps(study_rf.best_params),
        "xgb_best_params": json.dumps(study_xgb.best_params),
        "n_features_raw": len(feature_cols),
        "n_numeric": len(numeric_cols),
        "n_categorical": len(categorical_cols),
    })

    tmp = df_test[["Patch", "Subsample", "Date", "Campaign"]].copy()
    tmp["window"] = w
    tmp["y_true"] = y_test.values
    tmp["y_pred_rf"] = yhat_rf
    tmp["y_pred_xgb"] = yhat_xgb
    tmp["y_pred_ens"] = yhat_ens
    tmp["unc_ens"] = unc_ens
    tmp["risk_class"] = risk_cls.astype(str)
    test_pred_rows.append(tmp)

    best_bundle_by_window[w] = {
        "feature_cols": feature_cols,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "rf": best_rf,
        "xgb": best_xgb,
        "meta": meta_model,
        "rf_best_params": study_rf.best_params,
        "xgb_best_params": study_xgb.best_params,
        "rf_best_value": study_rf.best_value,
        "xgb_best_value": study_xgb.best_value,
    }

print("\n✅ BLOCK H COMPLETE — all windows processed.")


In [ ]:
# ======================================
# BLOCK I. SELECT BEST WINDOW (UNCHANGED)
# ======================================

results_df = pd.DataFrame(results_rows).sort_values("ens_rmse")
test_pred_df = pd.concat(test_pred_rows, ignore_index=True)
meta_w_df = pd.DataFrame(meta_weights)
cv_df = pd.DataFrame(cv_log)

BEST_WINDOW = results_df.iloc[0]["window"]

print("\n" + "="*80)
print("BLOCK I — BEST WINDOW SELECTION")
print("Best window by ENS RMSE:", BEST_WINDOW)
# Compute and display MAE per window alongside RMSE and R2
results_df_display = results_df.copy()
for idx, row in results_df_display.iterrows():
    w = row['window']
    sub = test_pred_df[test_pred_df['window'] == w]
    if len(sub) == 0:
        continue
    results_df_display.loc[idx, 'rf_mae']  = mean_absolute_error(sub['y_true'], sub['y_pred_rf'])
    results_df_display.loc[idx, 'xgb_mae'] = mean_absolute_error(sub['y_true'], sub['y_pred_xgb'])
    results_df_display.loc[idx, 'ens_mae'] = mean_absolute_error(sub['y_true'], sub['y_pred_ens'])

print(results_df_display[[
    'window',
    'rf_rmse','rf_mae','rf_r2',
    'xgb_rmse','xgb_mae','xgb_r2',
    'ens_rmse','ens_mae','ens_r2',
    'auc_extreme','precision_extreme'
]])
print("="*80)

best_test = test_pred_df[test_pred_df["window"] == BEST_WINDOW].copy()
best_test["Year"] = pd.to_datetime(best_test["Date"]).dt.year

print("\nBest window test-set summary:")
print(best_test[["y_true","y_pred_ens","unc_ens"]].describe())


In [ ]:
# ======================================
# BLOCK J. GLOBAL + PER-YEAR METRICS (RF, XGB, ENSEMBLE)
# ======================================

risk_year_rows = []

print("\n" + "="*80)
print("BLOCK J — GLOBAL + PER-YEAR METRICS (BEST WINDOW:", BEST_WINDOW, ")")
print("="*80)

# Define models and their prediction columns
model_specs = [
    ("rf",   "y_pred_rf",   "RF"),
    ("xgb",  "y_pred_xgb",  "XGBoost"),
    ("ens",  "y_pred_ens",  "Ensemble"),
]

y_true_all = best_test["y_true"].values

# ---- Global metrics for each model
for key, y_col, label in model_specs:
    y_pred_all = best_test[y_col].values

    rmse_all, mae_all, r2_all = regression_metrics(y_true_all, y_pred_all)

    risk_cls_all, q_pred_all = risk_class_from_pred(y_pred_all)
    auc_all, prec_all, q75_true_all = auc_precision_extreme(
        y_true_all, y_pred_all, q_pred_all["q75"]
    )

    print(
        f"GLOBAL TEST ({label}) — "
        f"RMSE={rmse_all:.3f}, MAE={mae_all:.3f}, R2={r2_all:.3f}, "
        f"AUC={auc_all:.3f}, Precision={prec_all:.3f}"
    )

# ---- Per-year metrics for each model
for yr, grp in best_test.groupby("Year"):
    if len(grp) < 10:
        continue

    y_t = grp["y_true"].values

    for key, y_col, label in model_specs:
        y_p = grp[y_col].values

        rmse_yr, mae_yr, r2_yr = regression_metrics(y_t, y_p)

        cls, q_pred = risk_class_from_pred(y_p)
        auc, prec, q75_true = auc_precision_extreme(y_t, y_p, q_pred["q75"])

        risk_year_rows.append({
            "window": BEST_WINDOW,
            "model": label,
            "Year": int(yr),
            "n_obs": len(grp),
            "RMSE": rmse_yr,
            "MAE": mae_yr,
            "R2": r2_yr,
            "AUC_extreme": auc,
            "Precision_extreme": prec,
            "true_q75": q75_true,
            "pred_q75": q_pred["q75"],
        })

risk_year_df = pd.DataFrame(risk_year_rows)

print("\nPer-year TEST metrics (all models):")
print(risk_year_df)


In [ ]:
# ======================================
# BLOCK J-2. PATCH-LEVEL RISK CLASS SUMMARY (OPTIONAL)
# ======================================

patch_risk_df = (
    best_test
    .groupby("Patch")
    .agg(
        dominant_risk=("risk_class", lambda x: x.value_counts().idxmax()),
        frac_extreme=("risk_class", lambda x: np.mean(x == "Extreme")),
        mean_pred=("y_pred_ens", "mean"),
        p90_pred=("y_pred_ens", lambda x: np.percentile(x, 90))
    )
    .reset_index()
)

print("\nPatch-level risk summary:")
print(patch_risk_df.head(10))


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Global matplotlib style for publication / multi-panel figures.
# Increase FS if text becomes too small after combining panels.
# ---------------------------------------------------------------
FS = 36  # base font size

plt.rcParams.update({
    "font.family":           "sans-serif",
    "font.size":             FS,
    "axes.titlesize":        FS + 4,
    "axes.labelsize":        FS,
    "xtick.labelsize":       FS,
    "ytick.labelsize":       FS,
    "legend.fontsize":       FS,
    "legend.title_fontsize": FS,
    "figure.titlesize":      FS + 6,
    "lines.linewidth":       5,
    "lines.markersize":      10,
    "axes.linewidth":        4,
    "xtick.major.width":     4,
    "ytick.major.width":     4,
    "xtick.major.size":      8,
    "ytick.major.size":      8,
    "axes.spines.top":       False,
    "axes.spines.right":     False,
})

print(f"Global matplotlib style applied (base font size = {FS})")
print("Tip: change FS above if text is too small after combining panels into a figure.")


In [ ]:
# ================================================================================
# BLOCK K — SHAP + MANUAL ALE (UPDATED: scientific ALE selection + Ensemble only)
# ================================================================================

print("\n" + "="*80)
print("BLOCK K — SHAP + MANUAL ALE (PIPELINE-SAFE, NUM+CAT)")
print("="*80)

bundle = best_bundle_by_window[BEST_WINDOW]
feature_cols_best     = bundle["feature_cols"]
numeric_cols_best     = bundle["numeric_cols"]
categorical_cols_best = bundle["categorical_cols"]

X_test_best = df_test[feature_cols_best].copy()

# ================================================================================
# 1) SHAP VALUES (PIPELINE-SAFE) — unchanged
# ================================================================================
print("\nComputing SHAP values...")

# ---------- XGBoost ----------
xgb_pipe   = bundle["xgb"]
xgb_model  = xgb_pipe.named_steps["model"]
X_xgb_proc = xgb_pipe.named_steps["prep"].transform(X_test_best)
xgb_feat_names = get_preprocessed_feature_names(xgb_pipe)

expl_xgb = shap.TreeExplainer(xgb_model)
shap_xgb = expl_xgb.shap_values(X_xgb_proc)

shap_xgb_df = pd.DataFrame({
    "feature":       xgb_feat_names,
    "mean_abs_shap": np.abs(shap_xgb).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

# ---------- RF ----------
rf_pipe   = bundle["rf"]
rf_model  = rf_pipe.named_steps["model"]
X_rf_proc = rf_pipe.named_steps["prep"].transform(X_test_best)
rf_feat_names = get_preprocessed_feature_names(rf_pipe)

expl_rf  = shap.TreeExplainer(rf_model)
shap_rf  = expl_rf.shap_values(X_rf_proc)
if isinstance(shap_rf, list):
    shap_rf = shap_rf[0]

shap_rf_df = pd.DataFrame({
    "feature":       rf_feat_names,
    "mean_abs_shap": np.abs(shap_rf).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

# ================================================================================
# 2) VARIABLE GROUP COLOR MAPPING + LABEL MAP — unchanged
# ================================================================================
import matplotlib.patches as mpatches

VAR_GROUPS = {
    "soil":    {"keywords": ["Sand","Silt","Clay","BD","TWI","Slope","DEM","Aspect",
                              "Ct_","Nt_"],
                "color": "#440154"},
    "hydro":   {"keywords": ["WC_30cm","WC_60cm","WC_90cm","S_30cm","S_60cm","S_90cm",
                              "dS_30","dS_60","dS_90","dS_total","R_profile"],
                "color": "#31688E"},
    "climate": {"keywords": ["Precipitation","Tmin","Tmax"],
                "color": "#35B779"},
    "remote":  {"keywords": ["NDVI","NDWI","NDRE","SAVI","NIR","Blue","Green","Red",
                              "B5","B6","B7","B8","SWIR"],
                "color": "#90D743"},
    "mgmt":    {"keywords": ["Fert_N","N_surplus","Crop","Campaign"],
                "color": "#FDE725"},
}

LABEL_MAP = {
    "Precipitation_sum_90d":       "Precipitation (cumulative, 90d)",
    "Tmin_mean_90d":               "T_min (mean, 90d)",
    "Tmax_mean_90d":               "T_max (mean, 90d)",
    "NDVI_mean_90d":               "NDVI (mean, 90d)",
    "NDWI_mean_90d":               "NDWI (mean, 90d)",
    "NDRE_mean_90d":               "NDRE (mean, 90d)",
    "NIR_mean_90d":                "NIR (mean, 90d)",
    "SWIR 1_mean_90d":             "SWIR-1 (mean, 90d)",
    "SWIR1_mean_90d":              "SWIR-1 (mean, 90d)",
    "WC_30cm_mean_90d":            "SM₃₀ (mean, 90d)",
    "WC_60cm_mean_90d":            "SM₆₀ (mean, 90d)",
    "WC_90cm_mean_90d":            "SM₉₀ (mean, 90d)",
    "dS_30cm_mm_mean_90d":         "ΔS₃₀ (mean, 90d)",
    "dS_60cm_mm_mean_90d":         "ΔS₆₀ (mean, 90d)",
    "dS_90cm_mm_mean_90d":         "ΔS₉₀ (mean, 90d)",
    "dS_total_mm_mean_90d":        "ΔS_total (mean, 90d)",
    "R_profile_mm_sum_90d":        "Drainage residual (cumulative, 90d)",
    "Silt":                        "Silt (%)",
    "Sand":                        "Sand (%)",
    "Clay":                        "Clay (%)",
    "Ct_0_30":                     "TC₀₋₃₀ (total carbon, 0–30 cm)",
    "Ct_30_60":                    "TC₃₀₋₆₀ (total carbon, 30–60 cm)",
    "Ct_60_90":                    "TC₆₀₋₉₀ (total carbon, 60–90 cm)",
    "Fert_N_curr_year":            "Fertilizer N (kg ha⁻¹)",
    "N_surplus":                   "N surplus (kg ha⁻¹)",
    "Crop_during_sampling_Rap":    "Crop: Oilseed rape",
    "Crop_during_sampling_Soats":  "Crop: Summer oats",
    "Crop_during_sampling_Lup":    "Crop: Lupine",
    "Crop_during_sampling_Whe":    "Crop: Winter wheat",
    "Crop_during_sampling_Maiz":   "Crop: Grain maize",
    "Crop_during_sampling_Bar":    "Crop: Winter barley",
    "Crop_during_sampling_Rye":    "Crop: Winter rye",
    "Crop_during_sampling_Soy":    "Crop: Soybean",
    "Crop_during_sampling_Sun":    "Crop: Sunflower",
    "Crop_during_sampling_Pha":    "Crop: Phacelia",
    "Crop_during_sampling_Woats":  "Crop: Winter oats",
}

def apply_label(raw_name):
    if raw_name in LABEL_MAP:
        return LABEL_MAP[raw_name]
    return raw_name.replace("_", " ")

def get_feature_color(feat_name):
    for grp, info in VAR_GROUPS.items():
        if any(kw in feat_name for kw in info["keywords"]):
            return info["color"]
    return "#546E7A"

# ================================================================================
# 3) SHAP BAR PLOTS — unchanged
# ================================================================================
SHAP_FIGSIZE = (26, 22)

def shap_bar_colored(shap_vals, feat_names, title, fpath,
                     figsize=(26, 22), top_n=15):
    mean_abs  = np.abs(shap_vals).mean(axis=0)
    order     = np.argsort(mean_abs)[::-1][:top_n]
    feats_raw = [feat_names[i] for i in order]
    feats     = [apply_label(f) for f in feats_raw]
    vals      = mean_abs[order]
    colors    = [get_feature_color(f) for f in feats_raw]

    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(range(len(feats)), vals[::-1], color=colors[::-1])
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats[::-1])
    ax.set_xlabel("Mean |SHAP value| (kg ha⁻¹)")
    ax.set_title(title)

    legend_patches = [mpatches.Patch(color=info["color"],
                                     label=grp.capitalize())
                      for grp, info in VAR_GROUPS.items()]
    legend_patches.append(mpatches.Patch(color="#546E7A", label="Other"))
    ax.legend(handles=legend_patches, loc="lower right",
              title="Variable group", framealpha=0.85)
    plt.tight_layout()
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  ✅ Colorized SHAP bar saved: {fpath}")

shap_bar_colored(
    shap_xgb, xgb_feat_names,
    title="XGBoost — SHAP Feature Importance (by Variable Group)",
    fpath=os.path.join(PLOTS_DIR, "shap_bar_xgb_colored.png"),
    figsize=SHAP_FIGSIZE
)
shap_bar_colored(
    shap_rf, rf_feat_names,
    title="RF — SHAP Feature Importance (by Variable Group)",
    fpath=os.path.join(PLOTS_DIR, "shap_bar_rf_colored.png"),
    figsize=SHAP_FIGSIZE
)
print("✅ SHAP importance computed + resized plots saved")

# ================================================================================
# 4) ENSEMBLE SHAP RANKING — derived automatically from best window
# ================================================================================
print("\n" + "-"*60)
print("Computing Ensemble SHAP ranking for ALE variable selection...")
print(f"Best window: {BEST_WINDOW}")

# Extract meta-learner coefficients automatically from best bundle
meta_coef_rf  = bundle["meta"].coef_[0]
meta_coef_xgb = bundle["meta"].coef_[1]
print(f"Meta-learner coefficients — RF: {meta_coef_rf:.4f}, "
      f"XGB: {meta_coef_xgb:.4f}")

# Warn and fallback if any coefficient is negative
# (negative coefficients make weighted SHAP unintuitive)
if meta_coef_rf < 0 or meta_coef_xgb < 0:
    print("WARNING: Negative meta-learner coefficient detected.")
    print("         Falling back to equal weighting (0.5 / 0.5).")
    meta_coef_rf  = 0.5
    meta_coef_xgb = 0.5

# Build ensemble SHAP ranking on numeric features only
# (ALE is not valid for one-hot encoded categorical dummies)
rf_shap_dict  = dict(zip(rf_feat_names,
                         np.abs(shap_rf).mean(axis=0)))
xgb_shap_dict = dict(zip(xgb_feat_names,
                          np.abs(shap_xgb).mean(axis=0)))

ens_shap_records = []
for feat in numeric_cols_best:
    rf_val  = rf_shap_dict.get(feat, 0.0)
    xgb_val = xgb_shap_dict.get(feat, 0.0)
    ens_val = meta_coef_rf * rf_val + meta_coef_xgb * xgb_val
    ens_shap_records.append({
        "feature":       feat,
        "mean_abs_shap": ens_val
    })

ens_shap_df = (
    pd.DataFrame(ens_shap_records)
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

print(f"\nEnsemble SHAP ranking — top 10 continuous predictors "
      f"(window = {BEST_WINDOW}):")
print(ens_shap_df.head(10).to_string(index=False))

# Select top 5 — the ALE selection criterion stated in M&M
top5_for_ale = ens_shap_df["feature"].tolist()[:5]
print(f"\nTop 5 selected for ALE (Ensemble SHAP, continuous only):")
for i, f in enumerate(top5_for_ale, 1):
    print(f"  {i}. {f}  →  {apply_label(f)}")

# ================================================================================
# 5) ALE FUNCTION — updated to return raw data for rug plot
# ================================================================================
def ale_1d(pipe, X, feature, grid=20):
    """
    1D ALE on original raw numeric feature.
    Returns: centers, ale_values, raw_data_for_rug
    Valid for continuous numeric features only.
    Reference: Apley and Zhu (2020).
    """
    x = X[feature].values
    if np.all(pd.isna(x)) or np.nanmin(x) == np.nanmax(x):
        return [], [], []

    x_non = x[~pd.isna(x)]
    bins  = np.unique(np.quantile(x_non, np.linspace(0, 1, grid + 1)))

    effects, centers = [], []
    for i in range(len(bins) - 1):
        lo, hi = bins[i], bins[i + 1]
        idx = (x >= lo) & (x <= hi)
        if idx.sum() < 5:
            continue
        X_lo = X.loc[idx].copy(); X_lo[feature] = lo
        X_hi = X.loc[idx].copy(); X_hi[feature] = hi
        effects.append(np.mean(pipe.predict(X_hi) - pipe.predict(X_lo)))
        centers.append((lo + hi) / 2)

    if len(effects) == 0:
        return [], [], []

    ale = np.cumsum(effects)
    ale -= ale.mean()
    return centers, ale, x_non   # x_non returned for rug plot

# ================================================================================
# 6) ENSEMBLE PIPE WRAPPER
# ================================================================================
class EnsemblePipe:
    """Thin wrapper — allows ale_1d to work on the stacked ensemble."""
    def __init__(self, rf_pipe, xgb_pipe, meta):
        self.rf   = rf_pipe
        self.xgb  = xgb_pipe
        self.meta = meta

    def predict(self, X):
        p_rf  = self.rf.predict(X)
        p_xgb = self.xgb.predict(X)
        return self.meta.predict(np.column_stack([p_rf, p_xgb]))

ens_pipe = EnsemblePipe(bundle["rf"], bundle["xgb"], bundle["meta"])

# ================================================================================
# 7) COMBINED ALE FIGURE — Ensemble only, top 5, manuscript-ready
#    Following Tamagno et al. (2022) style:
#    - ALE line + directional fill
#    - Zero reference line
#    - Rug marks showing data distribution (x-axis)
#    - Manuscript variable labels
#    - Panel letters inside each panel
# ================================================================================
print("\nGenerating combined ALE figure (Ensemble, top 5)...")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for idx, feature in enumerate(top5_for_ale):
    ax = axes_flat[idx]

    result = ale_1d(ens_pipe, X_test_best, feature)
    xs, ale_vals, x_raw = result

    if len(xs) == 0:
        ax.text(0.5, 0.5, f"No valid bins\n{apply_label(feature)}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=9, color="grey")
        continue

    xs       = np.array(xs)
    ale_vals = np.array(ale_vals)

    # ALE line
    ax.plot(xs, ale_vals,
            color="#6A1B9A", lw=2.5, marker="o", ms=10, zorder=3)

    # Directional fill — positive (purple) / negative (red)
    ax.fill_between(xs, ale_vals, 0,
                    where=(ale_vals >= 0),
                    alpha=0.15, color="#6A1B9A", zorder=2)
    ax.fill_between(xs, ale_vals, 0,
                    where=(ale_vals < 0),
                    alpha=0.15, color="#C62828", zorder=2)

    # Zero reference line
    ax.axhline(0, linestyle="--", color="black", lw=1.2, zorder=1)

    # Rug plot — data distribution on x-axis (Tamagno et al. style)
    ax.plot(x_raw,
            np.zeros_like(x_raw),
            "|", color="grey", alpha=0.5, ms=8,
            transform=ax.get_xaxis_transform(),
            zorder=4)

    # Axis labels
    ax.set_xlabel(apply_label(feature), fontsize=20)
    ax.set_ylabel("ALE (kg NO₃⁻ ha⁻¹)", fontsize=20)

    # Panel letter — top left inside panel
    panel_letter = chr(ord("a") + idx)
    ax.text(0.04, 0.95, f"({panel_letter})",
            transform=ax.transAxes,
            fontsize=20, fontweight="bold",
            va="top", ha="left")

    ax.tick_params(axis="both", labelsize=20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused 6th panel
axes_flat[5].set_visible(False)

fig.suptitle(
    f"Accumulated Local Effects — Ensemble Model "
    f"({BEST_WINDOW}-day aggregation window)",
    fontsize=20, fontweight="bold", y=1.01
)
plt.tight_layout()

ale_out = os.path.join(PLOTS_DIR, "ale_ensemble_top5_combined.png")
plt.savefig(ale_out, dpi=300, bbox_inches="tight")
plt.close()

print(f"✅ Combined ALE figure saved: {ale_out}")
print(f"   Window: {BEST_WINDOW}d | Variables: {top5_for_ale}")
print(f"   Meta weights — RF: {meta_coef_rf:.4f}, XGB: {meta_coef_xgb:.4f}")


In [ ]:
# ======================================
# BLOCK L. OBSERVED vs PREDICTED SCATTER (RF, XGB, ENSEMBLE) — ENHANCED
# ======================================

print("\n" + "="*80)
print("BLOCK L — ENHANCED OBSERVED vs PREDICTED SCATTER PLOTS")
print("="*80)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# ── Attach crop metadata: merge on Patch + Year using Crop_during_sampling
if "Crop_during_sampling" in df_test.columns:
    patch_crop = df_test[["Patch", "Crop_during_sampling"]].copy()
    patch_crop["Year"] = pd.to_datetime(df_test["Date"]).dt.year
    best_test_aug = best_test.merge(patch_crop, on=["Patch", "Year"], how="left")
else:
    best_test_aug = best_test.copy()
    best_test_aug["Crop_during_sampling"] = "Unknown"

# ── Yield zone: high vs low based on median patch-mean observed NO3
patch_means = best_test.groupby("Patch")["y_true"].mean()
median_val  = patch_means.median()
yield_zone_map = {p: ("High-yield" if v >= median_val else "Low-yield")
                  for p, v in patch_means.items()}
best_test_aug["yield_zone"] = best_test_aug["Patch"].map(yield_zone_map)

# ── Crop colors: automatically built from unique crop names in the data
CROP_NAMES  = sorted(best_test_aug["Crop_during_sampling"].dropna().unique())
n_crops     = len(CROP_NAMES)
crop_cmap   = plt.cm.get_cmap("viridis", n_crops)
CROP_COLOR_MAP = {crop: crop_cmap(i) for i, crop in enumerate(CROP_NAMES)}

# ── Yield zone marker shapes
ZONE_MARKER = {"High-yield": "o", "Low-yield": "^"}   # circle vs triangle
ZONE_COLORS = {"High-yield": "#31688E", "Low-yield": "#FDE725"}   # kept for Figure 1

print(f"Crops found ({n_crops}):", CROP_NAMES)
print("Yield zone counts:\n", best_test_aug["yield_zone"].value_counts())

# ── Use best_test (pre-merge, no duplicates) for all metric calculations
STATS_BASE = best_test.copy()

# ── Shared axis limits across all models (for comparability)
all_true = STATS_BASE["y_true"].values
all_pred = pd.concat([STATS_BASE[c] for c in ["y_pred_rf","y_pred_xgb","y_pred_ens"]]).values
pad  = (max(all_true.max(), all_pred.max()) - min(all_true.min(), all_pred.min())) * 0.05
LIMS = [min(all_true.min(), all_pred.min()) - pad,
        max(all_true.max(), all_pred.max()) + pad]

MODEL_SPECS = [
    ("y_pred_rf",  "RF",       "#440154"),
    ("y_pred_xgb", "XGBoost",  "#31688E"),
    ("y_pred_ens", "Ensemble", "#35B779"),
]


def _add_stats(ax, y_col):
    y_true = STATS_BASE["y_true"].values
    y_pred = STATS_BASE[y_col].values
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    ax.text(0.97, 0.05,
            f"R² = {r2:.2f}\nRMSE = {rmse:.2f}\nMAE = {mae:.2f}",
            transform=ax.transAxes, ha="right", va="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.80))

def _format_ax(ax, title):
    ax.plot(LIMS, LIMS, "k--", lw=2.5)
    ax.set_xlim(LIMS); ax.set_ylim(LIMS)
    ax.set_xlabel("Observed NO₃⁻ (kg ha⁻¹)")
    ax.set_ylabel("Predicted NO₃⁻ (kg ha⁻¹)")
    ax.set_title(title)
    ax.set_aspect("equal")   # square axes — 1:1 scale as reviewers requested


# ════════════════════════════════════════════════════════════════
# FIGURE 1 — colored by yield zone
# ════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(50, 20))
for ax, (y_col, label, _) in zip(axes, MODEL_SPECS):
    for zone, grp in best_test_aug.groupby("yield_zone"):
        ax.scatter(grp["y_true"], grp[y_col],
                   c=ZONE_COLORS.get(zone, "grey"),
                   s=280, alpha=0.60, edgecolors="k", linewidths=0.8,
                   label=zone, zorder=3)
    _format_ax(ax, f"{label} — by Yield Zone")
    _add_stats(ax, y_col)
    ax.legend(loc="upper left", framealpha=0.85)

fig.suptitle(f"Observed vs Predicted — by Yield Zone (Best Window: {BEST_WINDOW})",
             fontweight="bold", y=1.01)
plt.tight_layout()
fname1 = f"scatter_by_yieldzone_{BEST_WINDOW}.png"
plt.savefig(os.path.join(PLOTS_DIR, fname1), dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ Saved: {fname1}")


# ════════════════════════════════════════════════════════════════
# FIGURE 2 — colored by year
# ════════════════════════════════════════════════════════════════
years     = sorted(best_test_aug["Year"].unique())
year_cmap = plt.cm.get_cmap("viridis", len(years))

fig, axes = plt.subplots(1, 3, figsize=(50, 20))
for ax, (y_col, label, _) in zip(axes, MODEL_SPECS):
    for i, yr in enumerate(years):
        grp = best_test_aug[best_test_aug["Year"] == yr]
        ax.scatter(grp["y_true"], grp[y_col],
                   c=[year_cmap(i)], s=280, alpha=0.60,
                   edgecolors="k", linewidths=0.8,
                   label=str(yr), zorder=3)
    _format_ax(ax, f"{label} — by Year")
    _add_stats(ax, y_col)
    ax.legend(loc="upper left", framealpha=0.85, title="Year")

fig.suptitle(f"Observed vs Predicted — by Year (Best Window: {BEST_WINDOW})",
             fontweight="bold", y=1.01)
plt.tight_layout()
fname2 = f"scatter_by_year_{BEST_WINDOW}.png"
plt.savefig(os.path.join(PLOTS_DIR, fname2), dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ Saved: {fname2}")


# ════════════════════════════════════════════════════════════════
# FIGURE 3 — colored by crop
# ════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(50, 20))
for ax, (y_col, label, _) in zip(axes, MODEL_SPECS):
    for crop in CROP_NAMES:
        grp = best_test_aug[best_test_aug["Crop_during_sampling"] == crop]
        if len(grp) == 0:
            continue
        ax.scatter(grp["y_true"], grp[y_col],
                   c=[CROP_COLOR_MAP[crop]],
                   s=280, alpha=0.60, edgecolors="k", linewidths=0.8,
                   label=crop, zorder=3)
    _format_ax(ax, f"{label} — by Crop")
    _add_stats(ax, y_col)
    ax.legend(loc="upper left", framealpha=0.85, title="Crop",
              ncol=2, fontsize=plt.rcParams["legend.fontsize"] - 4)

fig.suptitle(f"Observed vs Predicted — by Crop (Best Window: {BEST_WINDOW})",
             fontweight="bold", y=1.01)
plt.tight_layout()
fname3 = f"scatter_by_crop_{BEST_WINDOW}.png"
plt.savefig(os.path.join(PLOTS_DIR, fname3), dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ Saved: {fname3}")


# ════════════════════════════════════════════════════════════════
# FIGURE 4 — color = crop type, shape = yield zone  ← NEW
# ════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(50, 20))

for ax, (y_col, label, _) in zip(axes, MODEL_SPECS):
    for crop in CROP_NAMES:
        for zone in ["High-yield", "Low-yield"]:
            grp = best_test_aug[
                (best_test_aug["Crop_during_sampling"] == crop) &
                (best_test_aug["yield_zone"] == zone)
            ]
            if len(grp) == 0:
                continue
            ax.scatter(
                grp["y_true"], grp[y_col],
                c=[CROP_COLOR_MAP[crop]],
                marker=ZONE_MARKER[zone],   # ● high-yield  ▲ low-yield
                s=310, alpha=0.65,
                edgecolors="k", linewidths=0.8,
                zorder=3
            )
    _format_ax(ax, f"{label} — Crop × Yield Zone")
    _add_stats(ax, y_col)

    # ── Legend: two sections ─────────────────────────────────────
    # Section 1 — crop colors
    crop_handles = [
        mpatches.Patch(facecolor=CROP_COLOR_MAP[c], edgecolor="k",
                       linewidth=0.6, label=c)
        for c in CROP_NAMES
    ]
    # Section 2 — yield zone shapes (neutral grey so color doesn't confuse)
    zone_handles = [
        mlines.Line2D([], [], color="grey", marker=ZONE_MARKER[z],
                      linestyle="None", markersize=20,
                      markeredgecolor="k", markeredgewidth=0.4,
                      label=z)
        for z in ["High-yield", "Low-yield"]
    ]

    # Combine with a blank separator entry
    blank = mpatches.Patch(color="none", label="")
    all_handles = crop_handles + [blank] + zone_handles

    ax.legend(
        handles=all_handles,
        loc="upper left", framealpha=0.5,
        ncol=3,
        fontsize=plt.rcParams["legend.fontsize"] - 1,
        handlelength=1.0,
    )

fig.suptitle(
    f"Observed vs Predicted — Crop × Yield Zone (Best Window: {BEST_WINDOW})",
    fontweight="bold", y=1.01
)
plt.tight_layout()
fname4 = f"scatter_crop_x_yieldzone_{BEST_WINDOW}.png"
plt.savefig(os.path.join(PLOTS_DIR, fname4), dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ Saved: {fname4}")

print("\n✅ All enhanced scatter plots saved.")

In [ ]:
# ======================================
# BLOCK M. ROC CURVES — GLOBAL + PER-YEAR (RF, XGB, ENSEMBLE)
# ======================================
from sklearn.metrics import roc_curve

print("\n" + "="*80)
print("BLOCK M — ROC CURVES (RF, XGB, ENSEMBLE) — BEST WINDOW:", BEST_WINDOW)
print("="*80)

# Redefine locally to ensure correct column order for this block
model_specs = [
    ("y_pred_rf",  "RF",       "rf"),
    ("y_pred_xgb", "XGBoost",  "xgb"),
    ("y_pred_ens", "Ensemble", "ens"),
]

def plot_roc(y_true, y_score, model_label, title, fname):
    q75_true = np.percentile(y_true, 75)
    y_bin = (y_true >= q75_true).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_score)
    plt.figure(figsize=(18, 16))
    plt.plot(fpr, tpr, label=f"{model_label}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, fname), dpi=300)
    plt.close()

# ---- Global ROC for RF, XGB, ENS
for y_col, label, short in model_specs:
    plot_roc(
        best_test["y_true"].values,
        best_test[y_col].values,
        model_label=label,
        title=f"Global ROC — {label} (Best Window {BEST_WINDOW})",
        fname=f"roc_global_{BEST_WINDOW}_{short}.png"
    )

# ---- Year-wise ROC for each model (n>=10)
for yr, grp in best_test.groupby("Year"):
    if len(grp) < 10:
        continue
    for y_col, label, short in model_specs:
        plot_roc(
            grp["y_true"].values,
            grp[y_col].values,
            model_label=label,
            title=f"ROC — {label} — Year {yr} (Best Window {BEST_WINDOW})",
            fname=f"roc_{yr}_{BEST_WINDOW}_{short}.png"
        )

print("✅ ROC plots for RF, XGBoost, and Ensemble saved to:", PLOTS_DIR)


In [ ]:
# ======================================
# BLOCK N. FINAL EXCEL EXPORT (UPDATED: consistent sheets)
# ======================================
print("\n" + "="*80)
print("BLOCK N — FINAL EXCEL EXPORT")
print("="*80)

# ── Add MAE columns to results_df before export
#    (computed from test_pred_df which has all window predictions)
results_df_export = results_df.copy()
for idx, row in results_df_export.iterrows():
    w   = row["window"]
    sub = test_pred_df[test_pred_df["window"] == w]
    if len(sub) == 0:
        continue
    results_df_export.loc[idx, "rf_mae"]  = mean_absolute_error(sub["y_true"], sub["y_pred_rf"])
    results_df_export.loc[idx, "xgb_mae"] = mean_absolute_error(sub["y_true"], sub["y_pred_xgb"])
    results_df_export.loc[idx, "ens_mae"] = mean_absolute_error(sub["y_true"], sub["y_pred_ens"])

# ── Reorder columns for readability
col_order = [
    "window",
    "rf_rmse",  "rf_mae",  "rf_r2",
    "xgb_rmse", "xgb_mae", "xgb_r2",
    "ens_rmse", "ens_mae", "ens_r2",
    "auc_extreme", "precision_extreme",
    "n_features_raw", "n_numeric", "n_categorical",
    "rf_best_params", "xgb_best_params",
]
col_order = [c for c in col_order if c in results_df_export.columns]
results_df_export = results_df_export[col_order]

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    results_df_export.to_excel(writer, sheet_name="window_summary", index=False)
    cv_df.to_excel(writer, sheet_name="cv_folds", index=False)
    meta_w_df.to_excel(writer, sheet_name="meta_weights", index=False)
    test_pred_df.to_excel(writer, sheet_name="test_predictions", index=False)
    risk_year_df.to_excel(writer, sheet_name="risk_by_year", index=False)
    shap_xgb_df.to_excel(writer, sheet_name="shap_xgb_best", index=False)
    shap_rf_df.to_excel(writer, sheet_name="shap_rf_best", index=False)

print("✅ All results written to:")
print(EXCEL_OUT)
print("\nWindow summary preview:")
print(results_df_export[["window","rf_rmse","rf_mae","rf_r2",
                          "xgb_rmse","xgb_mae","xgb_r2",
                          "ens_rmse","ens_mae","ens_r2"]].to_string(index=False))


In [ ]:
# ======================================
# BLOCK O. PATCH-LEVEL DIAGNOSTIC MAPS (UNCHANGED)
# ======================================

print("\n" + "="*80)
print("BLOCK N — PATCH-LEVEL DIAGNOSTIC MAPS (NO UNCERTAINTY + AUTO-ZOOM)")
print("="*80)

import matplotlib
matplotlib.use("Agg")

with zipfile.ZipFile(BOUNDARY_ZIP, "r") as zf:
    zf.extractall(os.path.join(OUT_DIR, "boundary"))

shp_files = [f for f in os.listdir(os.path.join(OUT_DIR, "boundary")) if f.endswith(".shp")]
if len(shp_files) == 0:
    raise FileNotFoundError("No shapefile found inside boundary zip.")

gdf_bound = gpd.read_file(os.path.join(OUT_DIR, "boundary", shp_files[0]))
gdf_bound["Patch"] = gdf_bound["Patch"].astype(str)

def build_patch_map(df_):
    patch_df = df_.groupby("Patch").agg(
        obs_kgNO3=("y_true", lambda x: np.percentile(x, 90)),
        pred_kgNO3=("y_pred_ens", lambda x: np.percentile(x, 90))
    ).reset_index()

    patch_df["residual"] = patch_df["pred_kgNO3"] - patch_df["obs_kgNO3"]
    return gdf_bound.merge(patch_df, on="Patch", how="inner")

gdf_map = build_patch_map(best_test)

vmin = min(gdf_map["obs_kgNO3"].min(), gdf_map["pred_kgNO3"].min())
vmax = max(gdf_map["obs_kgNO3"].max(), gdf_map["pred_kgNO3"].max())
res_max = np.max(np.abs(gdf_map["residual"]))

from matplotlib.colors import Normalize, TwoSlopeNorm
obs_norm = Normalize(vmin=vmin, vmax=vmax)
res_norm = TwoSlopeNorm(vcenter=0, vmin=-res_max, vmax=res_max)

xmin, ymin, xmax, ymax = gdf_bound.total_bounds
pad_x = (xmax - xmin) * 0.10
pad_y = (ymax - ymin) * 0.10
ZOOM_EXTENT = (xmin - pad_x, xmax + pad_x, ymin - pad_y, ymax + pad_y)

def plot_patch_maps(gdf_, title_suffix, fname):
    fig, axes = plt.subplots(1, 3, figsize=(24, 12))

    gdf_.plot(column="obs_kgNO3", ax=axes[0], legend=True, norm=obs_norm)
    axes[0].set_title("Observed")

    gdf_.plot(column="pred_kgNO3", ax=axes[1], legend=True, norm=obs_norm)
    axes[1].set_title("Predicted")

    gdf_.plot(column="residual", ax=axes[2], legend=True, norm=res_norm)
    axes[2].set_title("Residual (Pred − Obs)")

    for ax in axes:
        ax.set_axis_off()
        ax.set_xlim(ZOOM_EXTENT[0], ZOOM_EXTENT[1])
        ax.set_ylim(ZOOM_EXTENT[2], ZOOM_EXTENT[3])

    plt.suptitle(f"Patch-level Diagnostics — {title_suffix}", fontsize=36)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, fname), dpi=300)
    plt.close()

plot_patch_maps(gdf_map,
                f"Best Window {BEST_WINDOW} (All Years)",
                f"map_overall_{BEST_WINDOW}.png")

for yr, grp in best_test.groupby("Year"):
    gdf_yr = build_patch_map(grp)
    plot_patch_maps(gdf_yr,
                    f"Year {yr} — Best Window {BEST_WINDOW}",
                    f"map_{yr}_{BEST_WINDOW}.png")

print("✅ Patch diagnostic maps saved in:", PLOTS_DIR)


# updated code for map plotting

In [ ]:
# ======================================
# BLOCK O. PATCH-LEVEL DIAGNOSTIC MAPS (MAX ZOOM TO MAPPED PATCHES)
# ======================================

print("\n" + "="*80)
print("BLOCK N — PATCH-LEVEL DIAGNOSTIC MAPS (NO UNCERTAINTY + MAX ZOOM)")
print("="*80)

import matplotlib
matplotlib.use("Agg")

with zipfile.ZipFile(BOUNDARY_ZIP, "r") as zf:
    zf.extractall(os.path.join(OUT_DIR, "boundary"))

shp_files = [f for f in os.listdir(os.path.join(OUT_DIR, "boundary")) if f.endswith(".shp")]
if len(shp_files) == 0:
    raise FileNotFoundError("No shapefile found inside boundary zip.")

gdf_bound = gpd.read_file(os.path.join(OUT_DIR, "boundary", shp_files[0]))
gdf_bound["Patch"] = gdf_bound["Patch"].astype(str)

def build_patch_map(df_):
    patch_df = df_.groupby("Patch").agg(
        obs_kgNO3=("y_true", lambda x: np.percentile(x, 90)),
        pred_kgNO3=("y_pred_ens", lambda x: np.percentile(x, 90))
    ).reset_index()

    patch_df["residual"] = patch_df["pred_kgNO3"] - patch_df["obs_kgNO3"]
    return gdf_bound.merge(patch_df, on="Patch", how="inner")

gdf_map = build_patch_map(best_test)

# ---- shared scaling ----
vmin = min(gdf_map["obs_kgNO3"].min(), gdf_map["pred_kgNO3"].min())
vmax = max(gdf_map["obs_kgNO3"].max(), gdf_map["pred_kgNO3"].max())
res_max = np.max(np.abs(gdf_map["residual"]))

from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec

obs_norm = Normalize(vmin=vmin, vmax=vmax)
res_norm = TwoSlopeNorm(vcenter=0, vmin=-res_max, vmax=res_max)

def plot_patch_maps(gdf_, title_suffix, fname):
    cmap_main = "viridis"
    cmap_res  = "coolwarm"

    # ---- compute zoom from ONLY the mapped patches in this figure ----
    xmin, ymin, xmax, ymax = gdf_.total_bounds
    pad_x = max((xmax - xmin) * 0.01, 1e-9)
    pad_y = max((ymax - ymin) * 0.01, 1e-9)
    xlim = (xmin - pad_x, xmax + pad_x)
    ylim = (ymin - pad_y, ymax + pad_y)

    # GridSpec: 3 map cols + 2 narrow colorbar cols
    # Layout: [Observed] [cbar_obs_pred] [Predicted] [Residual] [cbar_res]
    fig = plt.figure(figsize=(21, 6.4))
    gs = GridSpec(
        1, 5,
        width_ratios=[1, 0.045, 1, 1, 0.045],
        wspace=0.08,
        left=0.01, right=0.99,
        top=0.87, bottom=0.06
    )

    ax0  = fig.add_subplot(gs[0, 0])   # Observed
    cax1 = fig.add_subplot(gs[0, 1])   # shared colorbar for Observed & Predicted
    ax1  = fig.add_subplot(gs[0, 2])   # Predicted
    ax2  = fig.add_subplot(gs[0, 3])   # Residual
    cax2 = fig.add_subplot(gs[0, 4])   # colorbar for Residual

    # ---- Observed ----
    gdf_.plot(
        column="obs_kgNO3",
        ax=ax0,
        cmap=cmap_main,
        norm=obs_norm,
        legend=False,
        edgecolor="black",
        linewidth=1.1
    )
    ax0.set_title("Observed", fontsize=17, pad=6)

    # ---- Predicted ----
    gdf_.plot(
        column="pred_kgNO3",
        ax=ax1,
        cmap=cmap_main,
        norm=obs_norm,
        legend=False,
        edgecolor="black",
        linewidth=1.1
    )
    ax1.set_title("Predicted", fontsize=17, pad=6)

    # ---- Residual ----
    gdf_.plot(
        column="residual",
        ax=ax2,
        cmap=cmap_res,
        norm=res_norm,
        legend=False,
        edgecolor="black",
        linewidth=1.1
    )
    ax2.set_title("Residual (Pred − Obs)", fontsize=17, pad=6)

    # ---- clean axes + max zoom ----
    for ax in (ax0, ax1, ax2):
        ax.set_axis_off()
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect("equal")

    # ---- colorbars ----
    sm_obs = ScalarMappable(norm=obs_norm, cmap=cmap_main)
    sm_obs.set_array([])

    sm_res = ScalarMappable(norm=res_norm, cmap=cmap_res)
    sm_res.set_array([])

    cbar1 = fig.colorbar(sm_obs, cax=cax1)
    cbar1.ax.tick_params(labelsize=13)

    cbar2 = fig.colorbar(sm_res, cax=cax2)
    cbar2.ax.tick_params(labelsize=13)

    fig.suptitle(f"Patch-level Diagnostics — {title_suffix}", fontsize=20, y=0.97)

    fig.savefig(os.path.join(PLOTS_DIR, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---- overall ----
plot_patch_maps(
    gdf_map,
    f"Best Window {BEST_WINDOW} (All Years)",
    f"map_overall_{BEST_WINDOW}.png"
)

# ---- year-wise ----
for yr, grp in best_test.groupby("Year"):
    gdf_yr = build_patch_map(grp)
    plot_patch_maps(
        gdf_yr,
        f"Year {yr} — Best Window {BEST_WINDOW}",
        f"map_{yr}_{BEST_WINDOW}.png"
    )

print("✅ Patch diagnostic maps saved in:", PLOTS_DIR)

# all years map in one figure

In [ ]:
# ======================================
# BLOCK O. PATCH-LEVEL DIAGNOSTIC MAPS (MAX ZOOM TO MAPPED PATCHES)
# ======================================

print("\n" + "="*80)
print("BLOCK O — PATCH-LEVEL DIAGNOSTIC MAPS (ZOOMED, NO DECORATIONS)")
print("="*80)

import matplotlib
matplotlib.use("Agg")
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Patch as MplPatch
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec

with zipfile.ZipFile(BOUNDARY_ZIP, "r") as zf:
    zf.extractall(os.path.join(OUT_DIR, "boundary"))

shp_files = [f for f in os.listdir(os.path.join(OUT_DIR, "boundary")) if f.endswith(".shp")]
if len(shp_files) == 0:
    raise FileNotFoundError("No shapefile found inside boundary zip.")

gdf_bound = gpd.read_file(os.path.join(OUT_DIR, "boundary", shp_files[0]))
gdf_bound["Patch"] = gdf_bound["Patch"].astype(str)

# ── Identify training vs test patches
train_patches = set(df_train["Patch"].astype(str).unique())
test_patches  = set(df_test["Patch"].astype(str).unique())

gdf_train_only = gdf_bound[gdf_bound["Patch"].isin(train_patches - test_patches)].copy()
gdf_test_bound = gdf_bound[gdf_bound["Patch"].isin(test_patches)].copy()
gdf_all_bound  = gdf_bound.copy()

def build_patch_map(df_):
    patch_df = df_.groupby("Patch").agg(
        obs_kgNO3=("y_true",      lambda x: np.percentile(x, 90)),
        pred_kgNO3=("y_pred_ens", lambda x: np.percentile(x, 90))
    ).reset_index()
    patch_df["Patch"]    = patch_df["Patch"].astype(str)
    patch_df["residual"] = patch_df["pred_kgNO3"] - patch_df["obs_kgNO3"]
    return gdf_bound.merge(patch_df, on="Patch", how="inner")

gdf_map = build_patch_map(best_test)

# ── Shared scaling
vmin    = min(gdf_map["obs_kgNO3"].min(),  gdf_map["pred_kgNO3"].min())
vmax    = max(gdf_map["obs_kgNO3"].max(),  gdf_map["pred_kgNO3"].max())
res_max = np.max(np.abs(gdf_map["residual"]))

obs_norm  = Normalize(vmin=vmin, vmax=vmax)
res_norm  = TwoSlopeNorm(vcenter=0, vmin=-res_max, vmax=res_max)
cmap_main = "viridis"
cmap_res  = "coolwarm"

# ── Zoom to TEST patches only — this is the key change
# Training patches are still drawn but do not define the zoom window
XMIN, YMIN, XMAX, YMAX = gdf_test_bound.total_bounds
MAP_W = XMAX - XMIN
MAP_H = YMAX - YMIN

# Minimal padding — just enough so edge patches are not clipped
pad = 0.03
XLIM = (XMIN - MAP_W * pad, XMAX + MAP_W * pad)
YLIM = (YMIN - MAP_H * pad, YMAX + MAP_H * pad)

# ── Font size
LABEL_FS = 11

# ── Patch centroids for labels
gdf_bound_c       = gdf_bound.copy()
gdf_bound_c["cx"] = gdf_bound_c.geometry.centroid.x
gdf_bound_c["cy"] = gdf_bound_c.geometry.centroid.y


def add_patch_labels(ax, fontsize=LABEL_FS):
    """Bold white labels for test patches only (within zoom window)."""
    for _, row in gdf_bound_c.iterrows():
        pid    = str(row["Patch"])
        cx, cy = row["cx"], row["cy"]
        # Only label patches within the zoom window
        if cx < XLIM[0] or cx > XLIM[1] or cy < YLIM[0] or cy > YLIM[1]:
            continue
        if pid in test_patches:
            ax.text(cx, cy, pid,
                    ha="center", va="center",
                    fontsize=fontsize, color="white", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.0,
                                                foreground="black")],
                    clip_on=True, zorder=5)
        else:
            ax.text(cx, cy, pid,
                    ha="center", va="center",
                    fontsize=max(fontsize - 3, 6),
                    color="#555555", fontweight="normal",
                    clip_on=True, zorder=5)


# ── Ordered row list
years      = sorted(best_test["Year"].unique())
row_labels = [f"Year {yr}" for yr in years] + ["Overall (All Years)"]
gdfs       = ([build_patch_map(best_test[best_test["Year"] == yr])
               for yr in years] + [gdf_map])
n_rows     = len(row_labels)

# ── Figure layout — taller rows now that decorations are removed
col_ratios     = [1, 1, 0.05, 1, 0.05]
row_h          = 3.5
legend_strip_h = 0.6
fig_h          = row_h * n_rows + legend_strip_h
fig_w          = 24
legend_frac    = legend_strip_h / fig_h

fig = plt.figure(figsize=(fig_w, fig_h))

gs = GridSpec(
    n_rows, 5,
    width_ratios=col_ratios,
    wspace=0.06, hspace=0.25,
    left=0.01, right=0.99,
    top=0.97,
    bottom=legend_frac + 0.01
)

# ── Shared colourbars
cax_obs = fig.add_subplot(gs[:, 2])
cax_res = fig.add_subplot(gs[:, 4])

sm_obs = ScalarMappable(norm=obs_norm, cmap=cmap_main); sm_obs.set_array([])
sm_res = ScalarMappable(norm=res_norm, cmap=cmap_res);  sm_res.set_array([])

cbar1 = fig.colorbar(sm_obs, cax=cax_obs)
cbar1.ax.tick_params(labelsize=16)
cbar1.set_label("kg NO₃⁻ ha⁻¹", fontsize=16, labelpad=6)

cbar2 = fig.colorbar(sm_res, cax=cax_res)
cbar2.ax.tick_params(labelsize=16)
cbar2.set_label("Residual (kg NO₃⁻ ha⁻¹)", fontsize=16, labelpad=6)

# ── Legend strip — training patch indicator only
ax_legend = fig.add_axes([0.01, 0.002, 0.98, legend_frac - 0.005])
ax_legend.set_axis_off()

legend_elements = [
    MplPatch(facecolor="white", edgecolor="black", linewidth=1.5,
             label="Training patch (not predicted)"),
]
ax_legend.legend(
    handles=legend_elements,
    loc="center", ncol=1,
    fontsize=16, frameon=True, framealpha=0.9,
    edgecolor="grey",
)

# ── Plot each row
for row_idx, (label, gdf_) in enumerate(zip(row_labels, gdfs)):

    ax_obs  = fig.add_subplot(gs[row_idx, 0])
    ax_pred = fig.add_subplot(gs[row_idx, 1])
    ax_res  = fig.add_subplot(gs[row_idx, 3])

    for ax in (ax_obs, ax_pred, ax_res):
        # Training patches — white fill, black outline
        if len(gdf_train_only) > 0:
            gdf_train_only.plot(ax=ax, facecolor="white",
                                edgecolor="black", linewidth=1.0, zorder=1)
        ax.set_xlim(XLIM)
        ax.set_ylim(YLIM)
        ax.set_aspect("equal")
        ax.set_axis_off()

    # Test patches with values
    gdf_.plot(column="obs_kgNO3",  ax=ax_obs,  cmap=cmap_main, norm=obs_norm,
              legend=False, edgecolor="black", linewidth=1.0, zorder=2)
    gdf_.plot(column="pred_kgNO3", ax=ax_pred, cmap=cmap_main, norm=obs_norm,
              legend=False, edgecolor="black", linewidth=1.0, zorder=2)
    gdf_.plot(column="residual",   ax=ax_res,  cmap=cmap_res,  norm=res_norm,
              legend=False, edgecolor="black", linewidth=1.0, zorder=2)

    # Patch labels
    for ax in (ax_obs, ax_pred, ax_res):
        add_patch_labels(ax)

    # Column headers (first row only)
    if row_idx == 0:
        ax_obs.set_title(f"{label}\nObserved",    fontsize=18,
                         fontweight="bold", loc="center", pad=4)
        ax_pred.set_title("Predicted",            fontsize=18,
                          fontweight="bold", loc="center", pad=4)
        ax_res.set_title("Residual (Pred − Obs)", fontsize=18,
                         fontweight="bold", loc="center", pad=4)
    else:
        ax_obs.set_title(label, fontsize=18, fontweight="bold",
                         loc="left", pad=4)

    # Dashed separator between rows
    if row_idx < n_rows - 1:
        fig.canvas.draw()
        pos    = ax_obs.get_position()
        y_line = pos.y0 - 0.003
        line   = Line2D(
            [0.01, 0.99], [y_line, y_line],
            transform=fig.transFigure,
            color="grey", linewidth=1.2, linestyle="--", alpha=0.6
        )
        fig.add_artist(line)

out_path = os.path.join(PLOTS_DIR, f"map_all_years_{BEST_WINDOW}.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print("✅ Combined patch diagnostic map saved:", out_path)

In [ ]:
# ======================================
# BLOCK P. PATCH-LEVEL DIAGNOSTIC + RISK MAPS (P90-based)
# ======================================

print("\n" + "="*80)
print("BLOCK O — PATCH-LEVEL DIAGNOSTICS + RISK CLASS MAPS (P90)")
print("="*80)

import matplotlib
matplotlib.use("Agg")

# ---- Load patch boundaries
with zipfile.ZipFile(BOUNDARY_ZIP, "r") as zf:
    zf.extractall(os.path.join(OUT_DIR, "boundary"))

shp_files = [f for f in os.listdir(os.path.join(OUT_DIR, "boundary")) if f.endswith(".shp")]
if len(shp_files) == 0:
    raise FileNotFoundError("No shapefile found inside boundary zip.")

gdf_bound = gpd.read_file(os.path.join(OUT_DIR, "boundary", shp_files[0]))
gdf_bound["Patch"] = gdf_bound["Patch"].astype(str)

# --------------------------------------------------
# Helper: build patch-level P90 table
# --------------------------------------------------
def build_patch_map(df_):
    patch_df = df_.groupby("Patch").agg(
        obs_kgNO3=("y_true", lambda x: np.percentile(x, 90)),
        pred_kgNO3=("y_pred_ens", lambda x: np.percentile(x, 90))
    ).reset_index()

    patch_df["residual"] = patch_df["pred_kgNO3"] - patch_df["obs_kgNO3"]
    return gdf_bound.merge(patch_df, on="Patch", how="inner")

# ---- Build global (all years) patch map
gdf_map = build_patch_map(best_test)

# --------------------------------------------------
# GLOBAL CONTINUOUS COLOR SCALES (P90)
# --------------------------------------------------
vmin = min(gdf_map["obs_kgNO3"].min(), gdf_map["pred_kgNO3"].min())
vmax = max(gdf_map["obs_kgNO3"].max(), gdf_map["pred_kgNO3"].max())
res_max = np.max(np.abs(gdf_map["residual"]))

from matplotlib.colors import Normalize, TwoSlopeNorm
obs_norm = Normalize(vmin=vmin, vmax=vmax)
res_norm = TwoSlopeNorm(vcenter=0, vmin=-res_max, vmax=res_max)

# --------------------------------------------------
# RISK CLASS DEFINITION (based on P90 across patches)
# --------------------------------------------------
q25, q50, q75 = np.percentile(gdf_map["pred_kgNO3"], [25, 50, 75])

def assign_risk_class(v):
    if v <= q25:
        return "Low"
    elif v <= q50:
        return "Moderate"
    elif v <= q75:
        return "High"
    else:
        return "Extreme"

gdf_map["risk_class"] = gdf_map["pred_kgNO3"].apply(assign_risk_class)

# Ordered categorical (important for plotting)
risk_order = ["Low", "Moderate", "High", "Extreme"]
gdf_map["risk_class"] = pd.Categorical(
    gdf_map["risk_class"], categories=risk_order, ordered=True
)

# Color palette for risk classes
risk_colors = {
    "Low": "#2ca25f",
    "Moderate": "#fee08b",
    "High": "#f46d43",
    "Extreme": "#a50026"
}

# --------------------------------------------------
# AUTO-ZOOM EXTENT
# --------------------------------------------------
xmin, ymin, xmax, ymax = gdf_bound.total_bounds
pad_x = (xmax - xmin) * 0.10
pad_y = (ymax - ymin) * 0.10
ZOOM_EXTENT = (xmin - pad_x, xmax + pad_x, ymin - pad_y, ymax + pad_y)

# --------------------------------------------------
# PLOT FUNCTION: P90 DIAGNOSTICS
# --------------------------------------------------
def plot_patch_diagnostics(gdf_, title_suffix, fname):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    gdf_.plot(column="obs_kgNO3", ax=axes[0], legend=True, norm=obs_norm)
    axes[0].set_title("Observed P90")

    gdf_.plot(column="pred_kgNO3", ax=axes[1], legend=True, norm=obs_norm)
    axes[1].set_title("Predicted P90")

    gdf_.plot(column="residual", ax=axes[2], legend=True, norm=res_norm)
    axes[2].set_title("Residual (Pred − Obs)")

    for ax in axes:
        ax.set_axis_off()
        ax.set_xlim(ZOOM_EXTENT[0], ZOOM_EXTENT[1])
        ax.set_ylim(ZOOM_EXTENT[2], ZOOM_EXTENT[3])

    plt.suptitle(f"Patch-level P90 Diagnostics — {title_suffix}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, fname), dpi=300)
    plt.close()

# --------------------------------------------------
# PLOT FUNCTION: RISK CLASS MAP
# --------------------------------------------------
def plot_risk_map(gdf_, title_suffix, fname):
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    gdf_.plot(
        column="risk_class",
        categorical=True,
        legend=True,
        ax=ax,
        color=[risk_colors[c] for c in gdf_["risk_class"]],
        legend_kwds={"title": "Risk class"}
    )

    ax.set_axis_off()
    ax.set_xlim(ZOOM_EXTENT[0], ZOOM_EXTENT[1])
    ax.set_ylim(ZOOM_EXTENT[2], ZOOM_EXTENT[3])

    plt.title(f"P90 Nitrate Risk Class — {title_suffix}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, fname), dpi=300)
    plt.close()

# --------------------------------------------------
# GLOBAL MAPS (ALL YEARS)
# --------------------------------------------------
plot_patch_diagnostics(
    gdf_map,
    f"Best Window {BEST_WINDOW} (All Years)",
    f"map_p90_diagnostics_overall_{BEST_WINDOW}.png"
)

plot_risk_map(
    gdf_map,
    f"Best Window {BEST_WINDOW} (All Years)",
    f"map_p90_risk_overall_{BEST_WINDOW}.png"
)

# --------------------------------------------------
# YEAR-WISE MAPS
# --------------------------------------------------
for yr, grp in best_test.groupby("Year"):
    gdf_yr = build_patch_map(grp)

    plot_patch_diagnostics(
        gdf_yr,
        f"Year {yr} — Best Window {BEST_WINDOW}",
        f"map_p90_diagnostics_{yr}_{BEST_WINDOW}.png"
    )

    # Use GLOBAL thresholds for consistency
    gdf_yr["risk_class"] = gdf_yr["pred_kgNO3"].apply(assign_risk_class)
    gdf_yr["risk_class"] = pd.Categorical(
        gdf_yr["risk_class"], categories=risk_order, ordered=True
    )

    plot_risk_map(
        gdf_yr,
        f"Year {yr} — Best Window {BEST_WINDOW}",
        f"map_p90_risk_{yr}_{BEST_WINDOW}.png"
    )

print("✅ P90 diagnostic maps + risk-class maps saved in:", PLOTS_DIR)
